In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

In [ ]:

def load_and_preprocess_data():
    url = "WineQT.csv"
    data = pd.read_csv(url, sep=';')
    
    X = data.drop('quality', axis=1).values
    y = data['quality'].values
    
    y = y - 3  
    
    
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
    
    X_train = torch.FloatTensor(X_train)
    y_train = torch.LongTensor(y_train)
    X_val = torch.FloatTensor(X_val)
    y_val = torch.LongTensor(y_val)
    X_test = torch.FloatTensor(X_test)
    y_test = torch.LongTensor(y_test)
    
    return X_train, y_train, X_val, y_val, X_test, y_test

X_train, y_train, X_val, y_val, X_test, y_test = load_and_preprocess_data()
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

In [ ]:
class ActivationFunctions:
    @staticmethod
    def relu(x, derivative=False):
        if derivative:
            return (x > 0).float()
        return torch.maximum(x, torch.zeros_like(x))
    
    @staticmethod
    def tanh(x, derivative=False):
        if derivative:
            return 1 - torch.tanh(x) ** 2
        return torch.tanh(x)
    
    @staticmethod
    def softmax(x):
        exp_x = torch.exp(x - torch.max(x, dim=1, keepdim=True)[0])
        return exp_x / torch.sum(exp_x, dim=1, keepdim=True)

In [ ]:
class LossFunctions:
    @staticmethod
    def categorical_cross_entropy(y_pred, y_true, derivative=False):
        batch_size = y_pred.shape[0]
        if derivative:
            grad = y_pred.clone()
            grad[range(batch_size), y_true] -= 1
            return grad / batch_size
        log_probs = torch.log(y_pred + 1e-8)
        loss = -log_probs[range(batch_size), y_true].mean()
        return loss

In [ ]:
class CustomMLP:
    def __init__(self, layer_sizes, activations, learning_rate=0.01):
        self.layer_sizes = layer_sizes  # [11, 64, 64, 6]
        self.activations = activations  # ['relu', 'relu', 'softmax']
        self.lr = learning_rate
        self.parameters = {}
        self.cache = {}
        
        # Ініціалізація Xavier/Glorot
        for i in range(1, len(layer_sizes)):
            limit = np.sqrt(6 / (layer_sizes[i-1] + layer_sizes[i]))
            W = torch.FloatTensor(layer_sizes[i], layer_sizes[i-1]).uniform_(-limit, limit)
            b = torch.zeros(layer_sizes[i], 1)
            self.parameters[f'W{i}'] = W
            self.parameters[f'b{i}'] = b
    
    def forward(self, X, training=True):
        self.cache = {'A0': X.T}  # (features, batch_size)
        A_prev = X.T
        
        for i in range(1, len(self.layer_sizes)):
            W = self.parameters[f'W{i}']
            b = self.parameters[f'b{i}']
            
            Z = W @ A_prev + b
            self.cache[f'Z{i}'] = Z
            
            if self.activations[i-1] == 'relu':
                A = ActivationFunctions.relu(Z)
            elif self.activations[i-1] == 'tanh':
                A = ActivationFunctions.tanh(Z)
            elif self.activations[i-1] == 'softmax':
                A = ActivationFunctions.softmax(Z.T).T  # (classes, batch)
            else:
                A = Z
            
            self.cache[f'A{i}'] = A
            A_prev = A
        
        return A.T  # повертаємо ймовірності (batch, classes)
    
    def backward(self, X, y_true):
        m = X.shape[0]
        y_pred = self.cache[f'A{len(self.layer_sizes)-1}'].T  # (classes, batch)
        
        # Градієнт для вихідного шару (softmax + cross-entropy)
        dZ = LossFunctions.categorical_cross_entropy(y_pred.T, y_true, derivative=True).T
        
        for i in reversed(range(1, len(self.layer_sizes))):
            A_prev = self.cache[f'A{i-1}']
            Z = self.cache[f'Z{i}']
            
            dW = (dZ @ A_prev.T) / m
            db = torch.sum(dZ, dim=1, keepdim=True) / m
            dA_prev = self.parameters[f'W{i}'].T @ dZ
            
            # Оновлення параметрів
            self.parameters[f'W{i}'] -= self.lr * dW
            self.parameters[f'b{i}'] -= self.lr * db
            
            # Градієнт для попереднього шару
            if i > 1:
                if self.activations[i-2] == 'relu':
                    dZ = dA_prev * ActivationFunctions.relu(Z, derivative=True)
                elif self.activations[i-2] == 'tanh':
                    dZ = dA_prev * ActivationFunctions.tanh(Z, derivative=True)
                else:
                    dZ = dA_prev
    
    def predict(self, X):
        with torch.no_grad():
            probs = self.forward(X, training=False)
            return torch.argmax(probs, dim=1)

In [ ]:
class PyTorchMLP(nn.Module):
    def __init__(self, input_size=11, hidden_sizes=[64, 64], output_size=6):
        super().__init__()
        layers = []
        prev_size = input_size
        for h_sizes = hidden_sizes + [output_size]
        
        for i, size in enumerate(h_sizes):
            layers.append(nn.Linear(prev_size, size))
            if i < len(h_sizes) - 1:
                layers.append(nn.ReLU())
            prev_size = size
        layers[-1] = nn.Identity()  # без softmax, бо в CrossEntropyLoss є вбудований
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

In [ ]:
def train_custom_mlp(model, X_train, y_train, X_val, y_val, epochs=1000):
    history = {'train_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        model.forward(X_train)
        loss = LossFunctions.categorical_cross_entropy(model.cache[f'A{len(model.layer_sizes)-1}'].T, y_train)
        model.backward(X_train, y_train)
        
        history['train_loss'].append(loss.item())
        
        if epoch % 100 == 0:
            with torch.no_grad():
                val_preds = model.predict(X_val)
                acc = accuracy_score(y_val.numpy(), val_preds.numpy())
                history['val_acc'].append(acc)
                print(f"Custom MLP | Epoch {epoch}: Train Loss = {loss:.4f}, Val Acc = {acc:.4f}")
    return history

In [ ]:

def train_custom_mlp(model, X_train, y_train, X_val, y_val, epochs=1000):
    history = {'train_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        model.forward(X_train)
        loss = LossFunctions.categorical_cross_entropy(model.cache[f'A{len(model.layer_sizes)-1}'].T, y_train)
        model.backward(X_train, y_train)
        
        history['train_loss'].append(loss.item())
        
        if epoch % 100 == 0:
            with torch.no_grad():
                val_preds = model.predict(X_val)
                acc = accuracy_score(y_val.numpy(), val_preds.numpy())
                history['val_acc'].append(acc)
                print(f"Custom MLP | Epoch {epoch}: Train Loss = {loss:.4f}, Val Acc = {acc:.4f}")
    return history

def train_pytorch_mlp(model, X_train, y_train, X_val, y_val, epochs=1000):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    history = {'train_loss': [], 'val_acc': []}
    
    for epoch in range(epochs):
        model.train()


In [ ]:
eters(), lr=0.01)
    history = {'train_loss': [], 'val_acc': []}
    
    for epoch in range(epochs):
        model.train()


In [ ]:
def train_custom_mlp(model, X_train, y_train, X_val, y_val, epochs=1000):
    history = {'train_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        model.forward(X_train)
        loss = LossFunctions.categorical_cross_entropy(model.cache[f'A{len(model.layer_sizes)-1}'].T, y_train)
        model.backward(X_train, y_train)
        
        history['train_loss'].append(loss.item())
        
        if epoch % 100 == 0:
            with torch.no_grad():
                val_preds = model.predict(X_val)
                acc = accuracy_score(y_val.numpy(), val_preds.numpy())
                history['val_acc'].append(acc)
                print(f"Custom MLP | Epoch {epoch}: Train Loss = {loss:.4f}, Val Acc = {acc:.4f}")
    return history

In [ ]:
def train_pytorch_mlp(model, X_train, y_train, X_val, y_val, epochs=1000):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    history = {'train_loss': [], 'val_acc': []}
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        
        history['train_loss'].append(loss.item())
        
        if epoch % 100 == 0:
            model.eval()
            with torch.no_grad():
                val_outputs = model(X_val)
                val_preds = torch.argmax(val_outputs, dim=1)
                acc = accuracy_score(y_val.numpy(), val_preds.numpy())
                history['val_acc'].append(acc)
                print(f"PyTorch MLP | Epoch {epoch}: Train Loss = {loss:.4f}, Val Acc = {acc:.4f}")
    return history

In [ ]:
# Запуск
custom_mlp = CustomMLP(layer_sizes=[11, 64, 64, 6], activations=['relu', 'relu', 'softmax'], learning_rate=0.01)
pytorch_mlp = PyTorchMLP()

print("Навчаємо власну реалізацію MLP...")
hist_custom = train_custom_mlp(custom_mlp, X_train, y_train, X_val, y_val, epochs=1000)

print("\nНавчаємо PyTorch модель...")
hist_pytorch = train_pytorch_mlp(pytorch_mlp, X_train, y_train, X_val, y_val, epochs=500)

# Оцінка на тесті
with torch.no_grad():
    custom_pred = custom_mlp.predict(X_test)
    pytorch_pred = torch.argmax(pytorch_mlp(X_test), dim=1)
    
    print("\nРезультати на тестовій вибірці:")
    print("Custom MLP Accuracy:", accuracy_score(y_test, custom_pred))
    print("PyTorch MLP Accuracy:", accuracy_score(y_test, pytorch_pred))